In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

base = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

for item in os.listdir(base):
    path = os.path.join(base, item)
    print(item, "→", "Folder" if os.path.isdir(path) else "File")

    if os.path.isdir(path):
        print("   Contents:", os.listdir(path)[:10])

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os
import json
import torch
import transformers
import peft

adapter_path = (
    "/kaggle/input/datasets/achintya2104/"
    "vima3ya-trained-adapters/VIMA3YA_assignment/"
    "fp16_lora_adapter"
)

with open(os.path.join(adapter_path, "adapter_config.json")) as f:
    config = json.load(f)

print("Base model:", config["base_model_name_or_path"])
print("LoRA rank:", config["r"])
print("Trainable adapter files:", os.listdir(adapter_path))
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

print("Base model loaded successfully!")
print("Model device:", next(base_model.parameters()).device)

In [ ]:
from peft import PeftModel

adapter_path = (
    "/kaggle/input/datasets/achintya2104/"
    "vima3ya-trained-adapters/VIMA3YA_assignment/"
    "fp16_lora_adapter"
)

model_16bit = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model_16bit.eval()

print("FP16 adapter attached successfully!")
print("Model device:", next(model_16bit.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y torchao

In [ ]:
import importlib.util

print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

In [ ]:
from peft import PeftModel

adapter_path = (
    "/kaggle/input/datasets/achintya2104/"
    "vima3ya-trained-adapters/VIMA3YA_assignment/"
    "fp16_lora_adapter"
)

model_16bit = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model_16bit.eval()

print("FP16 adapter attached successfully!")
print("Device:", next(model_16bit.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json
import os

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

partial_path = os.path.join(
    data_dir, "fp16_dev_predictions_partial.jsonl"
)

dev_path = os.path.join(data_dir, "dev.jsonl")

with open(partial_path, encoding="utf-8") as f:
    fp16_dev_predictions = [
        json.loads(line) for line in f if line.strip()
    ]

with open(dev_path, encoding="utf-8") as f:
    dev_data = [
        json.loads(line) for line in f if line.strip()
    ]

print("Previous predictions:", len(fp16_dev_predictions))
print("Total dev examples:", len(dev_data))
print("First saved prediction:", fp16_dev_predictions[0])

In [ ]:
# Check recovered predictions
saved_ids = {p["id"] for p in fp16_dev_predictions}
all_ids = {r["id"] for r in dev_data}

print("Recovered:", len(saved_ids))
print("Remaining:", len(all_ids - saved_ids))
print("Duplicate IDs:", len(fp16_dev_predictions) - len(saved_ids))
print("Last recovered ID:", fp16_dev_predictions[-1]["id"])

In [ ]:
import json
import os
import torch
from tqdm.auto import tqdm

output_path = "/kaggle/working/fp16_dev_predictions.jsonl"

# Save the 9 recovered predictions first
with open(output_path, "w", encoding="utf-8") as f:
    for item in fp16_dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

saved_ids = {p["id"] for p in fp16_dev_predictions}

# Generate only missing predictions
for record in tqdm(dev_data):
    if record["id"] in saved_ids:
        continue

    messages = record["messages"][:2]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    device = model_16bit.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_16bit.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    item = {
        "id": record["id"],
        "output": prediction
    }

    # Save immediately after each prediction
    with open(output_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

    saved_ids.add(record["id"])

print("\nCompleted predictions:", len(saved_ids))
print("Saved file:", output_path)


In [ ]:
import subprocess

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

!python {data_dir}/score.py \
    --gold {data_dir}/dev.jsonl \
    --pred /kaggle/working/fp16_dev_predictions.jsonl

In [ ]:
import subprocess

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

predictions = {
    "Base model": "base_dev_predictions.jsonl",
    "Schema rules": "rules_dev_predictions.jsonl",
    "Cleaned QLoRA": "qlora_dev_predictions.jsonl",
    "Raw QLoRA": "raw_dev_predictions.jsonl",
    "FP16 LoRA": "/kaggle/working/fp16_dev_predictions.jsonl"
}

for name, filename in predictions.items():
    pred_path = (
        filename if filename.startswith("/")
        else f"{data_dir}/{filename}"
    )

    print("\n", "=" * 45)
    print(name)
    print("=" * 45)

    subprocess.run([
        "python", f"{data_dir}/score.py",
        "--gold", f"{data_dir}/dev.jsonl",
        "--pred", pred_path
    ])

In [ ]:
import json
import os

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

files = [
    "base_dev_predictions.jsonl",
    "rules_dev_predictions.jsonl",
    "qlora_dev_predictions.jsonl",
    "raw_dev_predictions.jsonl",
    "fp16_dev_predictions_partial.jsonl"
]

for filename in files:
    path = os.path.join(data_dir, filename)

    with open(path, encoding="utf-8") as f:
        first = json.loads(next(f))

    print("\nFile:", filename)
    print("Keys:", list(first.keys()))
    print("ID:", first.get("id"))
    print("Prediction preview:", str(first)[:250])

In [ ]:
import json
import os

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

for source, destination in [
    ("qlora_dev_predictions.jsonl", "qlora_dev_scoring.jsonl"),
    ("raw_dev_predictions.jsonl", "raw_dev_scoring.jsonl")
]:
    input_path = os.path.join(data_dir, source)
    output_path = f"/kaggle/working/{destination}"

    with open(input_path, encoding="utf-8") as fin:
        rows = [json.loads(line) for line in fin if line.strip()]

    with open(output_path, "w", encoding="utf-8") as fout:
        for row in rows:
            fout.write(json.dumps({
                "id": row["id"],
                "output": row["prediction"]
            }) + "\n")

    print(destination, "saved:", len(rows))

In [ ]:
import subprocess

for name in ["qlora", "raw"]:
    print("\n", name.upper())

    subprocess.run([
        "python",
        f"{data_dir}/score.py",
        "--gold", f"{data_dir}/dev.jsonl",
        "--pred", f"/kaggle/working/{name}_dev_scoring.jsonl"
    ])

In [ ]:
import json
import torch
import os

data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

train_path = os.path.join(data_dir, "clean_train.jsonl")

with open(train_path, encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("Training record ID:", first_record.get("id"))
print("Message roles:", [
    msg["role"] for msg in first_record["messages"]
])
print("Training records:", sum(1 for _ in open(train_path)))

print("\nGPU memory:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 2),
        "GB allocated"
    )

In [ ]:
import gc
import torch

# Release the previous FP16 model
del model_16bit
del base_model

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared:")
print(
    "GPU 0 allocated:",
    round(torch.cuda.memory_allocated(0) / 1024**3, 2),
    "GB"
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map={"": 0}
)

print("Fresh base model loaded!")
print("Device:", next(base_model.parameters()).device)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config_r8 = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_r8 = get_peft_model(base_model, lora_config_r8)

model_r8.print_trainable_parameters()

In [ ]:
record = train_data[0]
messages = record["messages"]

full_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

prompt_ids = tokenizer.apply_chat_template(
    messages[:-1],
    tokenize=True,
    add_generation_prompt=True
)

print("Full token length:", len(full_ids))
print("Prompt token length:", len(prompt_ids))

for i, (a, b) in enumerate(zip(full_ids, prompt_ids)):
    if a != b:
        print("First mismatch position:", i)
        print("Full token:", a)
        print("Prompt token:", b)
        break
else:
    print("No mismatch in overlapping tokens.")

In [ ]:
print("SYSTEM MESSAGE:")
print(train_data[0]["messages"][0]["content"][:500])

print("\nUSER MESSAGE:")
print(train_data[0]["messages"][1]["content"][:500])

print("\nASSISTANT MESSAGE:")
print(train_data[0]["messages"][2]["content"][:500])

print("\nTOKENIZER CHAT TEMPLATE:")
print(tokenizer.chat_template[:500])

In [ ]:
print("Tokenizer type:", type(tokenizer))

print("\nFull IDs type:", type(full_ids))
print("Full IDs:", full_ids)

print("\nPrompt IDs type:", type(prompt_ids))
print("Prompt IDs:", prompt_ids)

print("\nFormatted conversation:")
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted[:1000])
print("\nFormatted length:", len(formatted))

In [ ]:
full_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)["input_ids"]

prompt_ids = tokenizer.apply_chat_template(
    messages[:-1],
    tokenize=True,
    add_generation_prompt=True
)["input_ids"]

print("Full token length:", len(full_ids))
print("Prompt token length:", len(prompt_ids))

print(
    "Prompt is exact prefix:",
    full_ids[:len(prompt_ids)] == prompt_ids
)

In [ ]:
from tqdm.auto import tqdm

train_tokens = []

for record in tqdm(train_data):
    messages = record["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    assert full_ids[:len(prompt_ids)] == prompt_ids

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    train_tokens.append({
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids)
    })

print("\nTraining examples:", len(train_tokens))
print("First example length:", len(train_tokens[0]["input_ids"]))
print(
    "Trainable response tokens:",
    sum(x != -100 for x in train_tokens[0]["labels"])
)

In [ ]:
from transformers import TrainingArguments

training_args_r8 = TrainingArguments(
    output_dir="/kaggle/working/experiment3_r8",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_torch",
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none"
)

print("Rank-8 training configuration ready!")
print("Epochs:", training_args_r8.num_train_epochs)
print("Batch size:", training_args_r8.per_device_train_batch_size)
print("Gradient accumulation:", training_args_r8.gradient_accumulation_steps)
print("Learning rate:", training_args_r8.learning_rate)

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class CustomCollator:
    tokenizer: object

    def __call__(self, features):
        batch = self.tokenizer.pad(
            [
                {
                    "input_ids": f["input_ids"],
                    "attention_mask": f["attention_mask"]
                }
                for f in features
            ],
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        max_len = batch["input_ids"].shape[1]

        labels = [
            f["labels"] + [-100] * (max_len - len(f["labels"]))
            for f in features
        ]

        batch["labels"] = torch.tensor(labels)
        return batch

collator_r8 = CustomCollator(tokenizer)

print("Custom collator ready!")

In [ ]:
from transformers import Trainer

trainer_r8 = Trainer(
    model=model_r8,
    args=training_args_r8,
    train_dataset=train_tokens,
    data_collator=collator_r8
)

print("Rank-8 Trainer created successfully!")

In [ ]:
import gc
import torch
from peft import LoraConfig, get_peft_model, TaskType

del trainer_r8
del model_r8
gc.collect()
torch.cuda.empty_cache()

lora_config_r8 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_r8 = get_peft_model(base_model, lora_config_r8)
model_r8.print_trainable_parameters()

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM

for name in ["trainer_r8", "model_r8", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map={"": 0}
)

print("Fresh base model loaded successfully!")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config_r8 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_r8 = get_peft_model(base_model, lora_config_r8)
model_r8.print_trainable_parameters()

In [ ]:
from transformers import Trainer

trainer_r8 = Trainer(
    model=model_r8,
    args=training_args_r8,
    train_dataset=train_tokens,
    data_collator=collator_r8
)

print("Rank-8 Trainer created successfully!")

In [ ]:
import gc
import torch
from peft import LoraConfig, get_peft_model, TaskType

del trainer_r8
del model_r8
gc.collect()
torch.cuda.empty_cache()

lora_config_r8 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_r8 = get_peft_model(base_model, lora_config_r8)
model_r8.print_trainable_parameters()

In [ ]:
training_args_r8.per_device_train_batch_size = 1
training_args_r8.gradient_accumulation_steps = 16

print("Batch size:", training_args_r8.per_device_train_batch_size)
print("Gradient accumulation:", training_args_r8.gradient_accumulation_steps)
print("Effective batch size:",
      training_args_r8.per_device_train_batch_size *
      training_args_r8.gradient_accumulation_steps)

In [ ]:
from transformers import Trainer

trainer_r8 = Trainer(
    model=model_r8,
    args=training_args_r8,
    train_dataset=train_tokens,
    data_collator=collator_r8
)

print("Rank-8 Trainer ready!")

In [ ]:
!nvidia-smi

import torch

for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}")
    print("Allocated:", round(torch.cuda.memory_allocated(i) / 1024**3, 2), "GB")
    print("Reserved:", round(torch.cuda.memory_reserved(i) / 1024**3, 2), "GB")

In [ ]:
print("Model device map:", getattr(base_model, "hf_device_map", "Not available"))
print("Model type:", type(model_r8))
print("Trainer processes:", trainer_r8.args.parallel_mode)
print("CUDA device count:", torch.cuda.device_count())

print("\nGPU memory before cleanup:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i)/1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i)/1024**3:.2f} GB reserved"
    )

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Unused cached GPU memory released.")

In [ ]:
!nvidia-smi

In [ ]:
print("Base model device:", next(base_model.parameters()).device)
print("LoRA model device:", next(model_r8.parameters()).device)

devices = {}
for name, param in model_r8.named_parameters():
    device = str(param.device)
    devices[device] = devices.get(device, 0) + param.numel()

print("\nParameter distribution:")
for device, count in devices.items():
    print(device, f"{count:,} parameters")

print("\nGPU allocated memory:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i)/1024**3:.2f} GB"
    )

In [ ]:
import gc
import torch

for name in ["trainer_r8", "model_r8", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print("Old training objects released.")

!nvidia-smi

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}")
    print("Allocated:", round(torch.cuda.memory_allocated(i) / 1024**3, 2), "GB")
    print("Reserved:", round(torch.cuda.memory_reserved(i) / 1024**3, 2), "GB")

In [ ]:
import os

important_files = [
    "/kaggle/working/fp16_dev_predictions.jsonl",
    "/kaggle/working/qlora_dev_scoring.jsonl",
    "/kaggle/working/raw_dev_scoring.jsonl"
]

for path in important_files:
    print(os.path.basename(path), ":", os.path.exists(path))

In [ ]:
import os
import shutil

backup_dir = "/kaggle/working/experiment_backup"
os.makedirs(backup_dir, exist_ok=True)

files_to_backup = [
    "/kaggle/working/fp16_dev_predictions.jsonl",
    "/kaggle/working/qlora_dev_scoring.jsonl",
    "/kaggle/working/raw_dev_scoring.jsonl"
]

for path in files_to_backup:
    if os.path.exists(path):
        shutil.copy2(path, backup_dir)

print("Backup files:")
print(os.listdir(backup_dir))

In [1]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

!nvidia-smi

CUDA available: True
Number of GPUs: 2
Thu Nov 12 06:22:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

In [2]:
data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

print("Dataset available:", os.path.exists(data_dir))
print("Files:", os.listdir(data_dir)[:10])

Dataset available: True
Files: ['base_dev_predictions.jsonl', 'clean_train.jsonl', 'cleaning_log.csv', 'dev.jsonl', 'experiment_results.csv', 'fp16_dev_predictions_partial.jsonl', 'qlora_dev_predictions.jsonl', 'raw_dev_predictions.jsonl', 'raw_train_backup.jsonl', 'rules_dev_predictions.jsonl']


In [3]:
import os
import gc
import json
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

print("Libraries loaded successfully!")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Libraries loaded successfully!
PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [4]:
data_dir = "/kaggle/input/datasets/achintya2104/vima3ya-llm-finetuning-assignment"

train_path = os.path.join(data_dir, "clean_train.jsonl")
dev_path = os.path.join(data_dir, "dev.jsonl")
test_path = os.path.join(data_dir, "test_inputs.jsonl")

print("Training file:", os.path.exists(train_path))
print("Development file:", os.path.exists(dev_path))
print("Test file:", os.path.exists(test_path))

Training file: True
Development file: True
Test file: True


In [5]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

with open(train_path, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

print("Tokenizer loaded!")
print("Training records:", len(train_data))
print("Padding token:", tokenizer.pad_token)

Tokenizer loaded!
Training records: 1445
Padding token: <|endoftext|>


In [6]:
train_tokens = []

for record in train_data:
    messages = record["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    assert full_ids[:len(prompt_ids)] == prompt_ids

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    train_tokens.append({
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids)
    })

print("Training tokens prepared:", len(train_tokens))
print("First example tokens:", len(train_tokens[0]["input_ids"]))

Training tokens prepared: 1445
First example tokens: 215


In [7]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map={"": 0}
)

base_model.config.use_cache = False

print("Fresh base model loaded!")
print("Device:", next(base_model.parameters()).device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fresh base model loaded!
Device: cuda:0


In [8]:
lora_config_r8 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_r8 = get_peft_model(base_model, lora_config_r8)
model_r8.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [10]:
model_r8.gradient_checkpointing_enable()
model_r8.enable_input_require_grads()

print("Gradient checkpointing enabled!")

Gradient checkpointing enabled!


In [11]:
training_args_r8 = TrainingArguments(
    output_dir="/kaggle/working/experiment3_r8",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_torch",
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [13]:
from dataclasses import dataclass

@dataclass
class CustomCollator:
    tokenizer: object

    def __call__(self, features):
        batch = self.tokenizer.pad(
            [
                {
                    "input_ids": f["input_ids"],
                    "attention_mask": f["attention_mask"]
                }
                for f in features
            ],
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        max_len = batch["input_ids"].shape[1]

        labels = [
            f["labels"] + [-100] * (max_len - len(f["labels"]))
            for f in features
        ]

        batch["labels"] = torch.tensor(labels)
        return batch

collator_r8 = CustomCollator(tokenizer)

print("Custom collator ready!")

Custom collator ready!


In [14]:
trainer_r8 = Trainer(
    model=model_r8,
    args=training_args_r8,
    train_dataset=train_tokens,
    data_collator=collator_r8
)

print("Rank-8 Trainer created successfully!")

Rank-8 Trainer created successfully!


In [15]:
import time
import torch

torch.cuda.reset_peak_memory_stats(0)
start_time_r8 = time.time()

train_result_r8 = trainer_r8.train()

training_minutes_r8 = (time.time() - start_time_r8) / 60
peak_gpu_gb_r8 = torch.cuda.max_memory_allocated(0) / (1024 ** 3)

print(f"Training time: {training_minutes_r8:.2f} minutes")
print(f"Peak GPU memory: {peak_gpu_gb_r8:.2f} GB")
print(f"Final training loss: {train_result_r8.training_loss:.6f}")

Step,Training Loss
20,0.348885
40,0.044749
60,0.032321
80,0.023288
100,0.016437
120,0.013072


Training time: 50.22 minutes
Peak GPU memory: 8.91 GB
Final training loss: 0.070799


In [16]:
import os

r8_save_path = "/kaggle/working/experiment3_r8/final_adapter"
os.makedirs(r8_save_path, exist_ok=True)

model_r8.save_pretrained(r8_save_path)
tokenizer.save_pretrained(r8_save_path)

print("Rank-8 adapter saved!")
print(os.listdir(r8_save_path))

Rank-8 adapter saved!
['README.md', 'tokenizer_config.json', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'chat_template.jinja']


In [17]:
import json
import torch
from tqdm.auto import tqdm

dev_path = os.path.join(data_dir, "dev.jsonl")
r8_predictions_path = "/kaggle/working/r8_dev_predictions.jsonl"

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = [json.loads(line) for line in f]

model_r8.eval()

with open(r8_predictions_path, "w", encoding="utf-8") as f:
    for record in tqdm(dev_data):
        messages = record["messages"][:2]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )

        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = model_r8.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False
            )

        prediction = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        f.write(json.dumps({
            "id": record["id"],
            "output": prediction
        }) + "\n")

print("Rank-8 predictions saved:", r8_predictions_path)
print("Total predictions:", len(dev_data))

  0%|          | 0/200 [00:00<?, ?it/s]

Rank-8 predictions saved: /kaggle/working/r8_dev_predictions.jsonl
Total predictions: 200


In [18]:
!python {data_dir}/score.py --gold {dev_path} --pred {r8_predictions_path}

group                            n   valid   exact  fields
all                            200  100.0%   73.5%   95.5%
category=account_access         19  100.0%   94.7%   99.3%
category=fraud                  15  100.0%   66.7%   94.2%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%   77.8%   96.5%
category=other                  16  100.0%   68.8%   96.1%
category=payment_failed         53  100.0%   67.9%   94.6%
category=refund                 55  100.0%   61.8%   93.0%

per-field accuracy (all rows)
  category        97.5%
  priority        83.0%
  amount_paise    95.5%
  txn_date        99.0%
  txn_id          99.5%
  channel         99.5%
  language       100.0%
  needs_human     90.0%


In [19]:
import gc
import torch
import os
import json

# Release Rank-8 model
for name in ["trainer_r8", "model_r8", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Locate saved Rank-16 adapter
adapter_path = (
    "/kaggle/input/datasets/achintya2104/"
    "vima3ya-trained-adapters/VIMA3YA_assignment/fp16_lora_adapter"
)

print("Rank-16 adapter exists:", os.path.exists(adapter_path))
print("Adapter files:", os.listdir(adapter_path))

# Check test dataset
with open(test_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

print("Test records:", len(test_data))

Rank-16 adapter exists: True
Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']
Test records: 400


In [20]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map={"": 0}
)

final_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

final_model.eval()

print("Final Rank-16 model loaded!")
print("Device:", next(final_model.parameters()).device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Final Rank-16 model loaded!
Device: cuda:0


In [21]:
from tqdm.auto import tqdm
import json
import os

test_output_path = "/kaggle/working/predictions_test.jsonl"

with open(test_output_path, "w", encoding="utf-8") as f:
    for record in tqdm(test_data):
        messages = record["messages"][:2]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )

        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = final_model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False
            )

        prediction = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        f.write(json.dumps({
            "id": record["id"],
            "output": prediction
        }) + "\n")

print("Test predictions saved!")
print("Total records:", len(test_data))
print("File exists:", os.path.exists(test_output_path))

  0%|          | 0/400 [00:00<?, ?it/s]

Test predictions saved!
Total records: 400
File exists: True


In [23]:
import json

required_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

valid_count = 0
invalid_count = 0
seen_ids = set()

with open(test_output_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            record = json.loads(line)
            prediction = json.loads(record["output"])

            assert set(prediction.keys()) == required_keys
            assert record["id"] not in seen_ids

            seen_ids.add(record["id"])
            valid_count += 1

        except Exception:
            invalid_count += 1

print("Valid predictions:", valid_count)
print("Invalid predictions:", invalid_count)
print("Unique IDs:", len(seen_ids))

Valid predictions: 400
Invalid predictions: 0
Unique IDs: 400


In [24]:
import json
import csv

gold_path = dev_path
pred_path = "/kaggle/working/fp16_dev_predictions.jsonl"

with open(gold_path, "r", encoding="utf-8") as f:
    gold_data = [json.loads(line) for line in f]

with open(pred_path, "r", encoding="utf-8") as f:
    pred_data = [json.loads(line) for line in f]

gold_by_id = {
    r["id"]: json.loads(r["messages"][-1]["content"])
    for r in gold_data
}

errors = []

for record in pred_data:
    pred = json.loads(record["output"])
    gold = gold_by_id[record["id"]]

    wrong_fields = [
        key for key in gold
        if pred.get(key) != gold[key]
    ]

    if wrong_fields:
        errors.append({
            "id": record["id"],
            "wrong_fields": wrong_fields,
            "gold": gold,
            "predicted": pred
        })

print("Total development errors:", len(errors))
print("\nFirst 10 error examples:\n")

for error in errors[:10]:
    print("\nID:", error["id"])
    for field in error["wrong_fields"]:
        print(
            field,
            "| Expected:", error["gold"][field],
            "| Predicted:", error["predicted"].get(field)
        )

Total development errors: 28

First 10 error examples:


ID: dv-00001
needs_human | Expected: False | Predicted: True

ID: dv-00006
txn_date | Expected: 2026-06-30 | Predicted: 2026-07-01

ID: dv-00007
amount_paise | Expected: 87700 | Predicted: 877

ID: dv-00011
priority | Expected: P1 | Predicted: P2
needs_human | Expected: True | Predicted: False

ID: dv-00012
priority | Expected: P1 | Predicted: P2
needs_human | Expected: True | Predicted: False

ID: dv-00022
amount_paise | Expected: 167700 | Predicted: 1677

ID: dv-00030
needs_human | Expected: False | Predicted: True

ID: dv-00034
priority | Expected: P2 | Predicted: P1
needs_human | Expected: False | Predicted: True

ID: dv-00037
priority | Expected: P1 | Predicted: P2
needs_human | Expected: True | Predicted: False

ID: dv-00042
category | Expected: other | Predicted: refund
channel | Expected: None | Predicted: card


In [25]:
import pandas as pd

error_rows = []

for error in errors:
    for field in error["wrong_fields"]:
        error_rows.append({
            "id": error["id"],
            "field": field,
            "expected": str(error["gold"][field]),
            "predicted": str(error["predicted"].get(field))
        })

error_df = pd.DataFrame(error_rows)

error_path = "/kaggle/working/error_analysis.csv"
error_df.to_csv(error_path, index=False)

print("Error analysis saved!")
print("Total field-level errors:", len(error_df))
print("\nErrors by field:")
print(error_df["field"].value_counts())

Error analysis saved!
Total field-level errors: 37

Errors by field:
field
needs_human     11
priority        10
amount_paise     6
category         4
txn_date         3
channel          2
txn_id           1
Name: count, dtype: int64


In [26]:
import pandas as pd

results = [
    {
        "Experiment": "Base model baseline",
        "Configuration": "Qwen2.5-1.5B",
        "Exact Match": 0.0,
        "Field Accuracy": 0.0,
        "Training Minutes": None,
        "Peak GPU GB": None
    },
    {
        "Experiment": "Schema rules baseline",
        "Configuration": "Qwen2.5-1.5B + rules prompt",
        "Exact Match": 0.0,
        "Field Accuracy": 0.0,
        "Training Minutes": None,
        "Peak GPU GB": None
    },
    {
        "Experiment": "Raw QLoRA",
        "Configuration": "Rank 16, 4-bit",
        "Exact Match": 71.5,
        "Field Accuracy": 95.1,
        "Training Minutes": 33.61,
        "Peak GPU GB": 11.25
    },
    {
        "Experiment": "Cleaned QLoRA",
        "Configuration": "Rank 16, 4-bit",
        "Exact Match": 80.5,
        "Field Accuracy": 96.7,
        "Training Minutes": 33.60,
        "Peak GPU GB": 10.38
    },
    {
        "Experiment": "Cleaned FP16 LoRA",
        "Configuration": "Rank 16, FP16",
        "Exact Match": 86.0,
        "Field Accuracy": 97.7,
        "Training Minutes": 25.66,
        "Peak GPU GB": 11.08
    },
    {
        "Experiment": "Rank-8 FP16 LoRA",
        "Configuration": "Rank 8, FP16",
        "Exact Match": 73.5,
        "Field Accuracy": 95.5,
        "Training Minutes": 50.22,
        "Peak GPU GB": 8.91
    }
]

results_df = pd.DataFrame(results)

results_path = "/kaggle/working/experiment_comparison.csv"
results_df.to_csv(results_path, index=False)

print(results_df.to_string(index=False))
print("\nSaved:", results_path)

           Experiment               Configuration  Exact Match  Field Accuracy  Training Minutes  Peak GPU GB
  Base model baseline                Qwen2.5-1.5B          0.0             0.0               NaN          NaN
Schema rules baseline Qwen2.5-1.5B + rules prompt          0.0             0.0               NaN          NaN
            Raw QLoRA              Rank 16, 4-bit         71.5            95.1             33.61        11.25
        Cleaned QLoRA              Rank 16, 4-bit         80.5            96.7             33.60        10.38
    Cleaned FP16 LoRA               Rank 16, FP16         86.0            97.7             25.66        11.08
     Rank-8 FP16 LoRA                Rank 8, FP16         73.5            95.5             50.22         8.91

Saved: /kaggle/working/experiment_comparison.csv


In [28]:
!pip -q install python-docx

In [29]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION

doc = Document()

section = doc.sections[0]
section.top_margin = Inches(0.55)
section.bottom_margin = Inches(0.55)
section.left_margin = Inches(0.65)
section.right_margin = Inches(0.65)

styles = doc.styles
styles["Normal"].font.name = "Arial"
styles["Normal"].font.size = Pt(9)

# Title
title = doc.add_heading(
    "NimbusPay Support Ticket Triage Using LoRA Fine-Tuning", 0
)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.add_run("VIMA3YA – Machine Learning Intern Assignment").bold = True

doc.add_heading("1. Objective", level=1)
doc.add_paragraph(
    "The objective was to fine-tune a small language model to classify "
    "NimbusPay customer support tickets and generate structured JSON "
    "containing category, priority, transaction details, language and "
    "human-escalation information."
)

doc.add_heading("2. Model and Methodology", level=1)
doc.add_paragraph(
    "Base model: Qwen2.5-1.5B-Instruct (approximately 1.5 billion parameters). "
    "Parameter-efficient fine-tuning was performed using LoRA and QLoRA. "
    "The LoRA configuration used rank 16, alpha 32, dropout 0.05 and "
    "the attention and MLP projection modules. The final selected model "
    "was the cleaned-data FP16 LoRA Rank-16 adapter, with 18,464,768 "
    "trainable parameters."
)
doc.add_paragraph(
    "Training used 3 epochs, learning rate 2e-4, effective batch size 16, "
    "response-only labels and gradient checkpointing. The FP16 experiment "
    "used AdamW. The QLoRA experiments used 4-bit quantization and "
    "paged AdamW 8-bit."
)

doc.add_heading("3. Data Preparation", level=1)
doc.add_paragraph(
    "The training dataset contained 1,445 records. Cleaning included "
    "repairing 25 invalid or truncated labels, correcting 70 legacy "
    "priority keys, 50 date and transaction-ID formats, 5 amount values, "
    "60 priority mismatches and 60 confirmed previous-contact labels. "
    "Structural validation reported zero errors after cleaning. "
    "A cleaning log was generated to document field-level changes."
)

doc.add_heading("4. Experimental Results", level=1)

table = doc.add_table(rows=1, cols=5)
table.style = "Light Shading Accent 1"
headers = ["Experiment", "Exact Match", "Field Accuracy",
           "Time (min)", "Peak GPU"]
for cell, value in zip(table.rows[0].cells, headers):
    cell.text = value

rows = [
    ["Raw QLoRA", "71.5%", "95.1%", "33.61", "11.25 GB"],
    ["Cleaned QLoRA", "80.5%", "96.7%", "33.60", "10.38 GB"],
    ["Cleaned FP16 LoRA R16", "86.0%", "97.7%", "25.66", "11.08 GB"],
    ["FP16 LoRA R8", "73.5%", "95.5%", "50.22", "8.91 GB"]
]

for row in rows:
    cells = table.add_row().cells
    for cell, value in zip(cells, row):
        cell.text = value

doc.add_paragraph(
    "The base-model and schema-rules prompt baselines both scored 0% "
    "under the strict evaluator because their generated outputs did not "
    "satisfy the required JSON format and schema. These scores indicate "
    "format-compliance failure, not necessarily zero classification ability."
)

doc.add_heading("5. Findings and Error Analysis", level=1)
doc.add_paragraph(
    "Cleaning improved QLoRA exact match from 71.5% to 80.5%. "
    "The cleaned FP16 Rank-16 model achieved the best development result: "
    "86.0% exact match and 97.7% mean field accuracy. It outperformed "
    "Rank-8 by 12.5 percentage points in exact match."
)
doc.add_paragraph(
    "The FP16 Rank-16 model produced 28 imperfect records and 37 "
    "field-level errors across 200 development examples. Error counts "
    "were: needs_human 11, priority 10, amount_paise 6, category 4, "
    "txn_date 3, channel 2 and txn_id 1. Amount conversion, priority "
    "classification and human escalation were the main error areas."
)

doc.add_heading("6. Final Test Inference", level=1)
doc.add_paragraph(
    "The selected FP16 Rank-16 adapter generated predictions for all "
    "400 test records. Inference used the first two messages, the model "
    "chat template, greedy generation (do_sample=False), max_new_tokens=200 "
    "and .strip() as the only output postprocessing. Validation confirmed "
    "400 valid JSON predictions, zero invalid records, 400 unique IDs "
    "and exactly eight required fields per prediction."
)

doc.add_heading("7. Limitations and Conclusion", level=1)
doc.add_paragraph(
    "The fourth proposed experiment was not completed and is not reported "
    "as a measured result. The Rank-8 run required a smaller micro-batch "
    "to fit available GPU memory. The final selected model was the cleaned "
    "FP16 Rank-16 adapter because it achieved the highest development "
    "exact-match and field-accuracy scores among the completed fine-tuned "
    "experiments. Future improvements should focus on priority rules, "
    "human-escalation decisions and amount normalization."
)

doc.add_paragraph(
    "Note: All reported metrics are development-set measurements unless "
    "otherwise stated. Test predictions were generated without using "
    "test labels."
)

report_path = "/kaggle/working/technical_report.docx"
doc.save(report_path)

print("Technical report created:", report_path)

Technical report created: /kaggle/working/technical_report.docx


In [30]:
readme_content = """
# NimbusPay Support Ticket Triage using LoRA Fine-Tuning

## Project Overview
Fine-tuned Qwen2.5-1.5B-Instruct to classify NimbusPay customer
support tickets and generate structured JSON predictions.

## Model
- Base model: Qwen2.5-1.5B-Instruct
- Fine-tuning: LoRA / QLoRA
- Best adapter: Cleaned FP16 LoRA, Rank 16
- Trainable parameters: 18,464,768
- LoRA alpha: 32
- LoRA dropout: 0.05

## Development Results

| Experiment | Exact Match | Field Accuracy |
|---|---:|---:|
| Raw QLoRA | 71.5% | 95.1% |
| Cleaned QLoRA | 80.5% | 96.7% |
| Cleaned FP16 LoRA Rank 16 | 86.0% | 97.7% |
| FP16 LoRA Rank 8 | 73.5% | 95.5% |

## Data Preparation
- Training records: 1,445
- Invalid labels repaired: 25
- Legacy priority keys corrected: 70
- Date and transaction ID formats corrected: 50
- Amount corrections: 5
- Priority mismatches corrected: 60
- Previous-contact labels corrected: 60

See `cleaning_log.csv` for the cleaning record.

## Final Test Predictions
- Test records: 400
- Valid JSON outputs: 400
- Invalid outputs: 0
- Unique IDs: 400

Output file: `predictions_test.jsonl`

## Error Analysis
The selected model had 28 imperfect development predictions
and 37 field-level errors. The most frequent errors were
needs_human and priority.

See `error_analysis.csv`.

## Repository Contents
- `adapter/` - Selected FP16 LoRA adapter
- `predictions_test.jsonl` - Final test predictions
- `cleaning_log.csv` - Training data cleaning record
- `error_analysis.csv` - Development error analysis
- `experiment_comparison.csv` - Experiment results
- `technical_report.docx` - Technical report
- `README.md` - Project documentation

## Limitations
The fourth proposed experiment was not completed and is not
reported as a measured result.

## Reproducibility
Use the saved adapter with Qwen2.5-1.5B-Instruct and the
same tokenizer/chat template. Follow the assignment's inference
requirements: two input messages, greedy generation,
max_new_tokens=200 and strip-only postprocessing.

Base model: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
"""

readme_path = "/kaggle/working/README.md"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content.strip())

print("README created:", readme_path)

README created: /kaggle/working/README.md


In [31]:
import os
import shutil

submission_dir = "/kaggle/working/VIMA3YA_Submission"
os.makedirs(submission_dir, exist_ok=True)

# Copy final deliverables
files_to_copy = {
    "/kaggle/working/README.md": "README.md",
    "/kaggle/working/predictions_test.jsonl": "predictions_test.jsonl",
    "/kaggle/working/cleaning_log.csv": "cleaning_log.csv",
    "/kaggle/working/error_analysis.csv": "error_analysis.csv",
    "/kaggle/working/experiment_comparison.csv": "experiment_comparison.csv",
    "/kaggle/working/technical_report.docx": "technical_report.docx"
}

for source, filename in files_to_copy.items():
    if os.path.exists(source):
        shutil.copy2(source, os.path.join(submission_dir, filename))
        print("Copied:", filename)
    else:
        print("MISSING:", source)

# Copy the selected FP16 Rank-16 adapter
shutil.copytree(
    adapter_path,
    os.path.join(submission_dir, "adapter"),
    dirs_exist_ok=True
)

print("\nSubmission folder contents:")
print(os.listdir(submission_dir))

Copied: README.md
Copied: predictions_test.jsonl
MISSING: /kaggle/working/cleaning_log.csv
Copied: error_analysis.csv
Copied: experiment_comparison.csv
Copied: technical_report.docx

Submission folder contents:
['README.md', 'predictions_test.jsonl', 'error_analysis.csv', 'technical_report.docx', 'adapter', 'experiment_comparison.csv']


In [32]:
import os
import shutil

cleaning_log_source = os.path.join(data_dir, "cleaning_log.csv")

cleaning_log_destination = os.path.join(
    submission_dir, "cleaning_log.csv"
)

if os.path.exists(cleaning_log_source):
    shutil.copy2(cleaning_log_source, cleaning_log_destination)
    print("Cleaning log copied successfully!")
else:
    print("Cleaning log not found!")

Cleaning log copied successfully!


In [33]:
for root, dirs, files in os.walk(submission_dir):
    for filename in files:
        path = os.path.join(root, filename)
        print(os.path.relpath(path, submission_dir),
              f"({os.path.getsize(path) / 1024:.1f} KB)")

cleaning_log.csv (18.5 KB)
README.md (2.0 KB)
predictions_test.jsonl (87.6 KB)
error_analysis.csv (1.1 KB)
technical_report.docx (37.7 KB)
experiment_comparison.csv (0.4 KB)
adapter/README.md (5.1 KB)
adapter/tokenizer_config.json (0.7 KB)
adapter/adapter_config.json (1.2 KB)
adapter/tokenizer.json (11154.2 KB)
adapter/adapter_model.safetensors (72178.8 KB)
adapter/chat_template.jinja (2.4 KB)


In [34]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/VIMA3YA_Submission",
    "zip",
    "/kaggle/working",
    "VIMA3YA_Submission"
)

print("ZIP created:", zip_path)

ZIP created: /kaggle/working/VIMA3YA_Submission.zip


In [35]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/VIMA3YA_Submission.zip"))

/kaggle/working/VIMA3YA_Submission.zip

In [36]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/VIMA3YA_Submission.zip'))

/kaggle/working/VIMA3YA_Submission.zip

In [37]:
import os

path = "/kaggle/working/VIMA3YA_Submission.zip"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    print("File size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

File exists: True
File size: 67.29 MB


In [38]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [39]:
import os

print(os.path.exists("/kaggle/working/VIMA3YA_Submission.zip"))

True


In [40]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/VIMA3YA_Submission.zip"))

/kaggle/working/VIMA3YA_Submission.zip

In [41]:
import os

print(os.path.exists("/kaggle/working/VIMA3YA_Submission.zip"))
print(os.path.getsize("/kaggle/working/VIMA3YA_Submission.zip") / (1024 * 1024), "MB")

True
67.29104042053223 MB


In [42]:

import getpass
import json
import shutil
import subprocess
from pathlib import Path

token = getpass.getpass("Enter your new Kaggle API token: ")

config_dir = Path.home() / ".kaggle"
config_dir.mkdir(exist_ok=True)
(config_dir / "access_token").write_text(token)
del token

export_dir = Path("/kaggle/working/vima3ya_export")
export_dir.mkdir(exist_ok=True)

shutil.copy2(
    "/kaggle/working/VIMA3YA_Submission.zip",
    export_dir / "VIMA3YA_Submission.zip"
)

metadata = {
    "title": "VIMA3YA Submission Export",
    "id": "achintya2104/vima3ya-submission-export",
    "licenses": [{"name": "other"}]
}

(export_dir / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(export_dir), "--private"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print("Upload command completed.")
else:
    print("Upload failed. Share the error message, but never share your token.")



Enter your new Kaggle API token:  ········



usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth} ...
kaggle: error: unrecognized arguments: --private

Upload failed. Share the error message, but never share your token.


In [43]:
import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "create",
     "-p", "/kaggle/working/vima3ya_export"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)


Starting upload for file VIMA3YA_Submission.zip
Upload successful: VIMA3YA_Submission.zip (67MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/achintya2104/vima3ya-submission-export


  0%|          | 0.00/67.3M [00:00<?, ?B/s]
 16%|█▋        | 11.0M/67.3M [00:00<00:00, 116MB/s]
 49%|████▊     | 32.7M/67.3M [00:00<00:00, 181MB/s]
 74%|███████▍  | 50.0M/67.3M [00:00<00:00, 171MB/s]
100%|██████████| 67.3M/67.3M [00:00<00:00, 103MB/s]

Return code: 0
